# Sesion 3 - Integracion con servicios en la nube (AWS / Google Cloud)

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps  
**Practica NoSQL - Integracion de Python y Procesamiento Paralelo**

## De que se trata esta sesion

Ya sabemos procesar datasets grandes con Dask (Fase 1) y almacenar resultados en una base de datos NoSQL con MongoDB (Fase 2). El último eslabón de un flujo de Big Data típico es la integración con servicios de almacenamiento en la nube, como **Amazon S3** (Amazon Simple Storage Service) o **Google Cloud Storage**.

Estos servicios permiten almacenar grandes cantidades de datos mediante **buckets**, que funcionan como contenedores de almacenamiento en la nube. Estos buckets pueden utilizarse para construir un **Data Lake**, un repositorio centralizado donde se almacenan tanto los datos crudos como los datos procesados y reportes, disponibles para diferentes procesos que los necesiten, como ETL, modelos de Machine Learning y dashboards de Business Intelligence.


En esta sesion vamos a:

1. Practicar boto3, el SDK (Software Development Kit) de AWS para Python, usando una **simulacion local** de S3
   con la libreria `moto`, para que puedas ejecutar todo el flujo sin necesidad de una
   cuenta de AWS real ni riesgo de costos.
2. Subir y descargar archivos.
3. Automatizar un respaldo (*backup*) de reportes, como haria un proceso real en
   produccion.
4. Revisar buenas practicas de seguridad con credenciales.
5. Cerrar la semana armando el **pipeline completo**: Dask procesa -> MongoDB
   almacena -> Cloud respalda.



## 1. Por que simular S3 con `moto` en esta practica

`boto3` es la libreria oficial de AWS para Python, y su sintaxis para subir o
descargar archivos es identica sin importar si el bucket es real o simulado. Por eso,
para la parte de aprendizaje en clase, vamos a usar `moto`: una libreria que
intercepta las llamadas de boto3 y las responde como si existiera un servidor de AWS
real, pero todo corre localmente, sin necesidad de una cuenta ni riesgo de generar
cargos.

Esto nos permite a todos ejecutar el mismo codigo sin depender de que cada quien
tenga (o no) una cuenta de AWS configurada. Todo lo que aprendas aqui aplica igual si
mas adelante te conectas a un bucket real (solo cambiarias como se crean las
credenciales, no la forma de usar boto3).  
<br>
**--> Boto3 es la herramienta que se comunica con AWS. Moto es la herramienta que simula ser AWS.**


In [ ]:
# Instalar boto3 y moto
!pip install boto3 moto


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.6 MB/s eta 0:00:00


In [ ]:
import os
import glob
from datetime import datetime

import pandas as pd
import boto3
from moto import mock_aws


## 2. Sintaxis basica de boto3: crear bucket, subir y descargar

`moto` funciona como un "interruptor": mientras estemos dentro de un bloque
`with mock_aws():`, todas las llamadas a boto3 se simulan localmente. Vamos a crear
un cliente de S3 y un bucket para nuestro proyecto.

**Importante:** en `moto`, la simulacion solo existe mientras el bloque `with` esta
activo. Por eso, en este notebook vamos a mantener el cliente y el bucket abiertos
durante toda la practica usando `mock_aws().start()` en vez de un bloque `with`, para
poder ir ejecutando celdas una por una.


In [ ]:
# Iniciamos la simulación de AWS, que permanecerá activa durante el resto del notebook
mock = mock_aws()
mock.start()

In [ ]:
# Creamos un cliente de S3 para interactuar con el servicio de almacenamiento
s3 = boto3.client("s3", region_name="us-east-1")


AWS region = una ubicación geográfica donde AWS cuenta con infraestructura o centros de datos.

````
us-east-1       → North Virginia, USA
us-west-2       → Oregon, USA
eu-west-1       → Ireland
sa-east-1       → São Paulo, Brazil
````
Una menor distancia física permite reducir la latencia, es decir, el tiempo que tardan los datos en viajar entre el usuario y los servidores.

In [ ]:
# Definimos el nombre del bucket que vamos a crear
NOMBRE_BUCKET = "bucket-auditoria-transacciones"
s3.create_bucket(Bucket=NOMBRE_BUCKET) # Creamos el bucket en la simulación de AWS

print(s3.list_buckets()["Buckets"]) # Mostramos los nombres de los buckets disponibles

[{'Name': 'bucket-auditoria-transacciones', 'CreationDate': datetime.datetime(2026, 9, 14, 21, 8, 21, tzinfo=tzlocal())}]


In [ ]:
# Definimos la carpeta local donde se encuentran los archivos
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# Subimos uno de los archivos que generamos en la Sesion 1

# Construimos la ruta del archivo local
archivo_local = os.path.join(DATA_DIR, "reporte_promedio_por_region.csv")
# Definimos la ruta y nombre del archivo dentro del bucket
archivo_remoto = "reportes/reporte_promedio_por_region.csv"

# Subimos el archivo local al bucket de S3
s3.upload_file(archivo_local, NOMBRE_BUCKET, archivo_remoto)

s3.list_objects_v2(Bucket=NOMBRE_BUCKET)

# Obtenemos y recorremos los archivos almacenados en el bucket
print("Archivos en el bucket:")
for obj in s3.list_objects_v2(Bucket=NOMBRE_BUCKET).get("Contents", []):
    print(" -", obj["Key"], f"({obj['Size']} bytes)")  # Mostramos el nombre y tamaño de cada archivo

Archivos en el bucket:
 - reportes/reporte_promedio_por_region.csv (312 bytes)


In [ ]:
# Descargamos el archivo para confirmar que el respaldo funciona correctamente

ruta_descarga = "reporte_descargado.csv"  # Definimos la ruta local donde guardaremos el archivo descargado
s3.download_file(NOMBRE_BUCKET, archivo_remoto, ruta_descarga)  # Descargamos el archivo desde el bucket de S3

print("Contenido descargado desde el bucket:")
pd.read_csv(ruta_descarga)

Contenido descargado desde el bucket:


,region,promedio_monto_alto_valor
0,La Libertad,5799.818216
1,San Salvador,5504.656368
2,San Miguel,5196.225108
3,Cuscatlan,5100.038060
4,Santa Ana,4802.109659
5,Sonsonate,4599.266547
6,Usulutan,4403.447877
7,La Paz,4302.411255
8,Ahuachapan,4098.452143
9,La Union,4002.629289



Acabamos de simulamar cómo un pipeline de datos real puede :  
generar un reporte → almacenarlo en la nube → recuperarlo posteriormente → verificar los datos recuperados.

```
        DATA PROCESSING
              │
              ↓
     reporte_promedio_por_region.csv
              │
              │ upload
              ↓
       ☁️ S3 BUCKET
    ┌─────────────────────┐
    │ reportes/           │
    │ └── reporte_....csv │
    └─────────────────────┘
              │
              │ download
              ↓
       Colab / Local
    reporte_descargado.csv
              │
              ↓
          Pandas

```

>**Nota:**  
> En este ejercicio utilizamos Moto (mock_aws()) para simular los servicios de AWS directamente desde Google Colab.  
>
> Como AWS está siendo simulado, estos recursos no aparecen en la consola real de AWS (no podemos ir a verificar en AWS website) y no estamos utilizando almacenamiento real de Amazon.  
>
> En un caso real, usando una cuenta de AWS y un bucket real, podríamos entrar a la AWS Management Console → S3 y visualizar el bucket y los archivos almacenados.

#### Comparación entre AWS/Google storage vs. OneDrive/Google Drive ?
Google Drive y OneDrive son plataformas de almacenamiento de archivos y colaboración en la nube. Amazon S3 es un servicio de almacenamiento de objetos en la nube, diseñado para ser utilizado de forma programática y servir como infraestructura para aplicaciones, pipelines de datos, copias de seguridad y Data Lakes.


## 3. Backup automatico de reportes

En un entorno real, este tipo de respaldo se automatiza: cada vez que se genera un
reporte nuevo, un script lo sube al bucket con un nombre que incluye la fecha, para
mantener un historial ordenado. Vamos a construir esa funcion.


In [ ]:
def backup_reporte_en_s3(dataframe, nombre_base, bucket):
    '''
    Guarda un DataFrame como CSV localmente y lo sube a S3 con un nombre
    que incluye la fecha del backup, dentro de una carpeta 'backups/'.
    '''
    nombre_archivo = f"{nombre_base}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"  # Generamos un nombre único usando la fecha y hora del backup
    dataframe.to_csv(nombre_archivo, index=False)  # Guardamos el DataFrame como un archivo CSV local

    try:
        s3.upload_file(nombre_archivo, bucket, f"backups/{nombre_archivo}")  # Subimos el archivo al bucket dentro de la carpeta backups/
        print(f"backup completado: backups/{nombre_archivo}")  # Confirmamos que el backup se realizó correctamente
    except Exception as error:
        print(f"Error al subir el backup a la nube: {error}")  # Mostramos el error si la subida falla
    finally:
        os.remove(nombre_archivo)  # Eliminamos el archivo local después de intentar subirlo


In [ ]:
# Cargamos el dataset de transacciones
df_transacciones = pd.read_csv(os.path.join(DATA_DIR, "muestra_transacciones_alto_riesgo.csv"))

# Generamos y almacenamos un respaldo en S3
backup_reporte_en_s3(df_transacciones, "backup_transacciones", NOMBRE_BUCKET)

Respaldo completado: backups/backup_transacciones_20260914_213937.csv


In [ ]:
s3.list_objects_v2(Bucket=NOMBRE_BUCKET)

print("Contenido actual del bucket:") # Obtenemos y recorremos los objetos almacenados en el bucket
for obj in s3.list_objects_v2(Bucket=NOMBRE_BUCKET).get("Contents", []):
    print(" -", obj["Key"])


Contenido actual del bucket:
 - backups/backup_transacciones_20260914_213937.csv
 - reportes/reporte_promedio_por_region.csv


## 4. Manejo seguro de credenciales

Cuando te conectes a una cuenta real de AWS o Google Cloud, necesitarás **credenciales** para autenticarte, como una *access key* y una *secret key*, o un archivo de credenciales de servicio.

Estas son algunas reglas que debes seguir siempre:

* **Nunca** escribas tus credenciales directamente en el código ni las subas a un repositorio de Git.
* Utiliza **environment variables** o archivos de configuración protegidos y excluidos de Git mediante `.gitignore`. Por ejemplo, puedes utilizar un archivo `.env` junto con la librería `python-dotenv`.
* Las **environment variables** permiten almacenar información sensible, como contraseñas y API keys, fuera del código fuente y acceder a ella desde Python cuando sea necesario.
* Si trabajas en equipo, cada persona debería utilizar sus propias credenciales y tener únicamente los permisos necesarios para realizar sus tareas. Esto se conoce como **principio de menor privilegio**.
* Si accidentalmente publicas una credencial en un repositorio, **revócala o desactívala inmediatamente** desde la plataforma correspondiente. Borrarla posteriormente del código no es suficiente, porque la credencial pudo haber quedado registrada en el historial del repositorio.


>.gitignore es un archivo que le indica a Git qué archivos o carpetas debe ignorar y no incluir en el repositorio, como credenciales, archivos temporales o datasets grandes.


### Ejemplo con environment variable

El siguiente código muestra cómo se leerían las credenciales desde variables de entorno en una conexión real a AWS:



In [ ]:
# Ejemplo de buena práctica para una cuenta real de AWS
# No es necesario ejecutarlo con la simulación de Moto

# import os
# s3_real = boto3.client(
#     "s3",
#     aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"], # Obtenemos la Access Key desde una variable de entorno
#     aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"], # Obtenemos la Secret Key desde una variable de entorno
#     region_name="us-east-1",
# )

print("Las credenciales reales se leen desde variables de entorno, nunca desde el código.")


> **Idea clave:** las credenciales son información sensible. El código puede compartirse; las credenciales no.

## 5. Errores comunes al trabajar con almacenamiento en la nube

**1. Credenciales expuestas**

nunca
hardcodees `AWS_ACCESS_KEY_ID` ni `AWS_SECRET_ACCESS_KEY` en tu codigo.

**2. Permisos de bucket (error 403 Forbidden)**

Intentar subir o leer archivos de un bucket sobre el cual tu usuario no tiene permisos
de escritura o lectura produce un error 403. Esto se soluciona revisando la politica
de permisos (IAM) asociada a tu usuario o rol.

**3. Nombres de bucket duplicados**

Los nombres de los buckets de S3 son globales (unicos en todo AWS, no solo en tu
cuenta). Si intentas crear un bucket con un nombre que ya esta en uso por otra
persona en el mundo, la creacion va a fallar. Por eso es comun agregar un sufijo
unico (tu nombre de empresa, un identificador aleatorio) al nombre del bucket.


## 6. Cerrando el flujo: pipeline completo Dask -> MongoDB -> Cloud

Ahora vamos a armar
una funcion que representa el pipeline completo.

1. **Dask** procesa el dataset grande y genera un reporte agregado.
2. **MongoDB** almacena ese reporte como documentos consultables.
3. **Cloud (S3)** guarda un respaldo del reporte para el equipo de auditoria.

Como ya generamos los resultados de Dask en la Sesion 1, aqui nos enfocamos en dejar
claro como se conectarian los tres pasos en un solo script. Si tu conexion a MongoDB
de la Sesion 2 sigue activa, puedes usarla aqui directamente; si no, esta funcion
se puede ejecutar solo con las partes de Dask y Cloud.


In [ ]:
import dask.dataframe as dd


def pipeline_completo(directorio_datos, bucket, mongo_uri=None):
    '''
    Ejecuta el flujo completo de la practica NoSQL:
    1. Procesa el dataset con Dask.
    2. (Opcional) Inserta el resultado en MongoDB si se recibe un mongo_uri.
    3. Sube un respaldo del resultado a Cloud Storage (S3 simulado con moto).
    '''
    # 1. Procesamiento con Dask
    df_dask = dd.read_csv(os.path.join(directorio_datos, "transacciones_2024_part*.csv"))
    reporte = (
        df_dask[df_dask["monto"] > 3000]
        .groupby("region")["monto"]
        .mean()
        .compute()
        .sort_values(ascending=False)
        .reset_index()
    )
    reporte.columns = ["region", "promedio_monto_alto_valor"]
    print(f"Paso 1 (Dask): reporte generado con {len(reporte)} regiones")

    # 2. Persistencia en MongoDB (opcional)
    if mongo_uri:
        from pymongo import MongoClient
        cliente_mongo = MongoClient(mongo_uri)
        coleccion = cliente_mongo["banco_db"]["reporte_pipeline_completo"]
        coleccion.delete_many({})
        coleccion.insert_many(reporte.to_dict(orient="records"))
        print(f"Paso 2 (MongoDB): {coleccion.count_documents({})} documentos insertados")
        cliente_mongo.close()
    else:
        print("Paso 2 (MongoDB): omitido, no se recibio mongo_uri")

    # 3. Backup en Cloud
    backup_reporte_en_s3(reporte, "reporte_pipeline_completo", bucket)
    print("Paso 3 (Cloud): respaldo completado")

    return reporte


resultado_pipeline = pipeline_completo(DATA_DIR, NOMBRE_BUCKET, mongo_uri=None)
resultado_pipeline


Si quieres probar el pipeline completo incluyendo MongoDB, vuelve a conectar tu
cliente de la Sesion 2 y pasa tu `MONGO_URI` como argumento:

```python
resultado_pipeline = pipeline_completo(DATA_DIR, NOMBRE_BUCKET, mongo_uri="mongodb://localhost:27017/")
```


In [ ]:
# Al terminar, detenemos la simulacion de AWS
mock.stop()
print("Simulacion de AWS detenida")


## 7. Si quieres conectar una cuenta real de AWS o Google Cloud

Todo lo que hicimos hoy con `moto` funciona identico contra una cuenta real, solo
cambia como obtienes las credenciales y el bucket.

Un resumen rapido de los pasos :

- **AWS**: crear una cuenta free tier, crear un usuario IAM con permisos limitados a
  S3, generar sus llaves de acceso, crear un bucket desde la consola o con
  `s3.create_bucket(...)`, y usar esas llaves en boto3 mediante variables de entorno.
- **Google Cloud**: crear una cuenta free tier, crear un proyecto, habilitar Cloud
  Storage, crear una cuenta de servicio y descargar su archivo de credenciales JSON,
  y usar la libreria `google-cloud-storage` (la logica de subir/descargar archivos es
  conceptualmente identica a boto3).


## Cierre de la sesion y de la semana

En estas tres sesiones armamos, pieza por pieza, un flujo completo de Big Data:

1. **Dask** para procesar un dataset que no cabe comodo en la memoria RAM,
   usando lazy evaluation y particiones.
2. **MongoDB** para almacenar los resultados con un modelo de datos flexible,
   y consultarlos con filtros, ordenamiento y agregaciones.
3. **Cloud Storage (S3)** para centralizar respaldos, simulando el rol de un
   Data Lake.

